In [ ]:
import requests
import zipfile

from io import BytesIO

import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

%matplotlib inline

## Data

We will return to our ERCOT load data.

Recall that this data represents the hourly energy demand in various regions of Texas.

We will aggregate it to daily totals and try to predict daily totals

In [ ]:
from bs4 import BeautifulSoup

base_url = "https://www.ercot.com/gridinfo/load/load_hist"

ercot_bs = BeautifulSoup(requests.get(base_url).text)

relevant_links = []
for link in ercot_bs.find_all("a"):
    link_text = link.text.lower()

    if ("hourly load data" in link_text) and ("archives" not in link_text):
        href = link.attrs["href"]

        year = href.split("/")[5]
        if int(year) > 2019:
            relevant_links.append(href)

load_dfs = []
for link in relevant_links:
    res = requests.get(link)
    zf = zipfile.ZipFile(BytesIO(res.content))
    load_dfs.append(
        pd.read_excel(
            BytesIO(zf.read(zf.filelist[0].filename)),
            engine="openpyxl"
        ).rename(
            columns={
                "HourEnding": "dt",
                "Hour Ending": "dt"
            }
        )
    )

In [ ]:
load = pd.concat(load_dfs)

In [ ]:
# Convert hours from 1-24 to 0-23
for i in range(24):
    old, new = f"{i+1:02}:00", f"{i:02}:00"
    load["dt"] = load["dt"].str.replace(old, new).str.replace("DST", "").str.strip()

In [ ]:
load["dt"] = pd.to_datetime(load["dt"])

load = load.dropna()

load_daily = load.set_index("dt").resample("D").sum()
load_hourly = load.set_index("dt")

In [ ]:
load_hourly.to_parquet("hourly_load_ercot.parquet")
load_daily.to_parquet("daily_load_ercot.parquet")

In [ ]:
load_hourly.sort_index().index

In [ ]:
%%timeit

pd.read_csv("hourly_load_ercot.csv")

In [ ]:
%%timeit

pd.read_parquet("hourly_load_ercot.parquet")

In [ ]:
load_hourly = load_hourly.reset.sort_index().dropna()

In [ ]:
foo = load_hourly.sort_index().dropna()

In [ ]:
# Sort the index first, then slice
foo = foo.sort_index()
result = foo.loc["2025-06-01":"2025-07-01", :]

In [ ]:
result.plot(y="ERCOT")